# Notebook 05 — Serialization, Serving Architecture, and Explainability

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('Project root:', project_root)


## Model Serialization

### Definition
Model Serialization in this project means we design, validate, serve, and observe ML predictions through stable HTTP interfaces.

### Theory
APIs decouple producers (model service) and consumers (web app, batch jobs, partner systems). Typed contracts reduce ambiguity and production failures.

### Motivation
Training and serving often run in different processes/machines; model artifact portability is required.

### Real-World Example
Daily retraining jobs publish signed artifacts consumed by online inference services.

### Visual Explanation
Diagram/code cell below shows component flow and responsibilities.

### Code Explanation
Code cells in this section are structured as: setup → implementation → validation output.

### Best Practices
- Keep contracts explicit and versioned.
- Validate early at boundary.
- Log request IDs for traceability.
- Measure latency, error rate, and throughput.

### Common Mistakes
- Loading untrusted pickle artifacts.
- Missing metadata (feature order/version/metrics) alongside model binary.


## Pickle vs Joblib vs ONNX

- Pickle: standard Python serialization, broad but unsafe with untrusted files.
- Joblib: efficient for NumPy/scikit-learn artifacts, common ML ops default.
- ONNX: cross-runtime portability, extra conversion complexity.

In [ ]:
import json
from pathlib import Path

from src.data import load_california_housing, split_dataset
from src.serialization import load_metadata
from src.training import train_best_model, build_metadata

X, y = load_california_housing()
split = split_dataset(X, y, random_state=42)
best, ranking = train_best_model(split.X_train, split.y_train, split.X_val, split.y_val, split.X_test, split.y_test)
meta = build_metadata(best, ranking, len(split.X_train), len(split.X_val), len(split.X_test))
print(json.dumps({k: meta[k] for k in ['model_name', 'rmse', 'r2', 'mape']}, indent=2))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
blocks = ['Client', 'REST API', 'Validation', 'Model Artifact', 'Prediction']
for i, b in enumerate(blocks):
    ax.text(i * 2.3, 0.5, b, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.3', fc='#f7f7d9', ec='#8a8a00'))
for i in range(len(blocks)-1):
    ax.annotate('', xy=(i*2.3+1.0, 0.5), xytext=(i*2.3+1.3, 0.5), arrowprops=dict(arrowstyle='->'))
plt.title('Production Inference Architecture')
plt.show()

## Explainable AI with SHAP

Use SHAP to expose feature contribution per prediction.
API endpoint `/explain` should return base value + per-feature contributions.